In [ ]:
import subprocess
import numpy as np
from numpy import linalg as LA
import matplotlib
import json

matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

In [ ]:
def get_data(bound, count, output="output.tmp",
             method="euler",
             omega=1.0, x0=1.0, v0=1.0):
    settings = {
        "output": output,
        "method": method,
        "bound": bound,
        "count": count,
        "omega": omega,
        "x0": x0,
        "v0": v0
    }
    with open("settings.json", "w") as f:
        json.dump(settings, f)

    res = subprocess.run(["./solver", "settings.json"])
    return np.loadtxt("output.tmp")

def lin(x, y):
    n = len(x)
    
    k = np.sum(x * y) / np.sum(x**2)
    
    residuals = y - k * x
    if n > 1:
        s2 = np.sum(residuals**2) / (n - 1)
    else:
        s2 = 0.0
    
    err_k = np.sqrt(s2 / np.sum(x**2))
    
    return k, err_k

def lin_(x, y):
    xy = x*y
    k = (np.mean(xy) - np.mean(x) * np.mean(y))/\
        (np.mean(x**2) - np.mean(x)**2)
    a = np.mean(y) - k*np.mean(x)

    sqrtn = (len(xy))**0.5
    err_k = 1/sqrtn * ((np.mean(y**2) - np.mean(y)**2)/\
                       (np.mean(x**2) - np.mean(x)**2) - k**2)**0.5
    err_a = err_k*(np.mean(x**2) - np.mean(x)**2)**0.5

    return k, err_k, a, err_a

In [ ]:
def get_properties(omega, x0, v0):
    phi = np.arctan(x0 * omega / v0)
    A = x0 / np.sin(phi)
    return phi, A

Properties

In [ ]:
x0 = 1.0
v0 = 1.0
omega = 1.0
phi, A = get_properties(omega, x0, v0)

def get_all(bound, count, method):
    data = get_data(bound, count, method=method,
                    omega=omega, x0=x0, v0=v0)
    t = data[:, 0]
    x = data[:, 1]
    v = data[:, 2]
    return t, x, v

Plot of energy

In [ ]:
method = "bw_eu"

bound = 100
count = 1000

t, x, v = get_all(bound, count, method)
plt.grid()
plt.plot(t, x*x + v*v/omega/omega)
plt.xlabel("$t$")
plt.ylabel(r"$E = x^2 + v^2/\omega^2$")
plt.show()

plt.close()

Phase diagrams

In [ ]:
method = "euler"

bound = 100
count = 1000

t, x, v = get_all(bound, count, method)
plt.grid()
plt.plot(x, v)
plt.xlabel("$x$")
plt.ylabel(r"$\dot{x}$")

plt.show()
plt.close()

Bounds of methods

Find approx by coordinate

In [ ]:
for m in ["euler", "bw_eu", "heun", "rk4"]:
    bound = 10.0
    dots = 25

    count = 1.25
    mult = np.sqrt(2)

    times = []
    errors = []
    for i in range(dots):
        dt = bound/count
        times.append(dt)

        t, x, v = get_all(bound, count, m)
        t = t[-1]
        x = x[-1]
        v = v[-1]
        errors.append(np.abs(A*np.sin(omega * t + phi) - x))

        count *= mult

    times = np.log(times)
    errors = np.log(errors)
    k, err_k, a, err_a = lin_(times, errors)
    print(m, k)
    plt.plot(times, errors, label=m)

plt.xlabel(r"$\ln{\Delta t}$")
plt.ylabel(r"$\ln{\text{Error}}$")
plt.legend()
plt.grid()
plt.show()

Find approx by energy

In [ ]:
for m in ["euler", "bw_eu", "heun", "rk4"]:
    bound = 10.0
    dots = 25

    count = 1.25
    mult = np.sqrt(2)

    times = []
    errors = []
    for i in range(dots):
        dt = bound/count
        times.append(dt)

        t, x, v = get_all(bound, count, m)
        x = x[-1]
        v = v[-1]
        errors.append(np.abs(A**2 - x**2 - v**2 / omega**2))

        count *= mult

    times = np.log(times)
    errors = np.log(errors)
    k, err_k, a, err_a = lin_(times, errors)
    print(m, k)
    plt.plot(times, errors, label=m)

plt.xlabel(r"$\ln{\Delta t}$")
plt.ylabel(r"$\ln{\text{Error}}$")
plt.legend()
plt.grid()
plt.show()